### Milestone 3: Model Build
#### Nicholas Stirling
#### DSC670-T301 Advanced Uses of Generative AI (2265-1)
#### Professor Frank Neugebauer
#### 5/10/26

This milestone will be the first fine-tuned model for the Customer Complaint Triage and Response project. The objective of the model is to classifyt customer complaints, summarize complaint narratives, and generate a professional response recommendation using a fine-tuned OpenAI model.

From a business and risk mitigation perspective, customer complaints represent a valuable source of operational and compliance information. Financial institutions receive large volumes of complaints across multiple service channels, and manually reviewing these complaints is inefficient and inconsistent. This project attempts to demonstrate how Generative AI can improve complaint triage, support operational efficiency, and reduce risk exposure through more standardized handling.

Below we will:
- Prepare our data
- Format training data for fine-tuning
- Upload the information to OpenAI
- Create a fine-tuning job
- Track the model build process
- Evaluate fine-tuning metrics
- Test the resulting model

In [1]:
import os
import json
import time
import requests
import pandas as pd

In [ ]:
OpenAI Key

In [3]:
complaints_df = pd.read_csv("Customer_support_data.csv")

complaints_df.head()

,Unique id,channel_name,category,Sub-category,Customer Remarks,Order_id,order_date_time,Issue_reported at,issue_responded,Survey_response_Date,Customer_City,Product_category,Item_price,connected_handling_time,Agent_name,Supervisor,Manager,Tenure Bucket,Agent Shift,CSAT Score
0,7e9ae164-6a8b-4521-a2d4-58f7c9fff13f,Outcall,Product Queries,Life Insurance,NaN,c27c9bb4-fa36-4140-9f1f-21009254ffdb,NaN,01/08/2023 11:13,01/08/2023 11:47,01-Aug-23,NaN,NaN,NaN,NaN,Richard Buchanan,Mason Gupta,Jennifer Nguyen,On Job Training,Morning,5
1,b07ec1b0-f376-43b6-86df-ec03da3b2e16,Outcall,Product Queries,Product Specific Information,NaN,d406b0c7-ce17-4654-b9de-f08d421254bd,NaN,01/08/2023 12:52,01/08/2023 12:54,01-Aug-23,NaN,NaN,NaN,NaN,Vicki Collins,Dylan Kim,Michael Lee,>90,Morning,5
2,200814dd-27c7-4149-ba2b-bd3af3092880,Inbound,Order Related,Installation/demo,NaN,c273368d-b961-44cb-beaf-62d6fd6c00d5,NaN,01/08/2023 20:16,01/08/2023 20:38,01-Aug-23,NaN,NaN,NaN,NaN,Duane Norman,Jackson Park,William Kim,On Job Training,Evening,5
3,eb0d3e53-c1ca-42d3-8486-e42c8d622135,Inbound,Returns,Reverse Pickup Enquiry,NaN,5aed0059-55a4-4ec6-bb54-97942092020a,NaN,01/08/2023 20:56,01/08/2023 21:16,01-Aug-23,NaN,NaN,NaN,NaN,Patrick Flores,Olivia Wang,John Smith,>90,Evening,5
4,ba903143-1e54-406c-b969-46c52f92e5df,Inbound,Cancellation,Not Needed,NaN,e8bed5a9-6933-4aff-9dc6-ccefd7dcde59,NaN,01/08/2023 10:30,01/08/2023 10:32,01-Aug-23,NaN,NaN,NaN,NaN,Christopher Sanchez,Austin Johnson,Michael Lee,0-30,Morning,5


We have now imported libraries we will use for data manipulation, connecting to OpenAI API and creating our fine-tuning job, and testing the model. The dataset that is loaded is a subset of customer complaint records that should be manageable for fine-tuning and processing time, that would increase with a larger dataset. Some of the key fields that will be used will be the category, sub-category, Customer Remarks, CSAT Score, Channel name, and Product_category. Customer Remarks field acts as the primary unstructured text input for the model. The goal is to transform these operational support records into supervised instruction-response examples sutiable for fine-tuning.  
The dataset was intended for customer service analytics rather than AI training so additional preprocessing and synthetic target generation were required.

In [4]:
complaints_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 85907 entries, 0 to 85906
Data columns (total 20 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Unique id                85907 non-null  object 
 1   channel_name             85907 non-null  object 
 2   category                 85907 non-null  object 
 3   Sub-category             85907 non-null  object 
 4   Customer Remarks         28742 non-null  object 
 5   Order_id                 67675 non-null  object 
 6   order_date_time          17214 non-null  object 
 7   Issue_reported at        85907 non-null  object 
 8   issue_responded          85907 non-null  object 
 9   Survey_response_Date     85907 non-null  object 
 10  Customer_City            17079 non-null  object 
 11  Product_category         17196 non-null  object 
 12  Item_price               17206 non-null  float64
 13  connected_handling_time  242 non-null    float64
 14  Agent_name            

In [5]:
complaints_df[['Customer Remarks', 'category', 'Sub-category']].head()

,Customer Remarks,category,Sub-category
0,NaN,Product Queries,Life Insurance
1,NaN,Product Queries,Product Specific Information
2,NaN,Order Related,Installation/demo
3,NaN,Returns,Reverse Pickup Enquiry
4,NaN,Cancellation,Not Needed


At this stage, I inspected the structure of the dataset to verify column names and identify missing values. With the model training highly reliant on high-quality examples instead of volume in this case, even small inconsistencies in labels or formatting can negatively affect model outputs. Because of this, I focused on removing null values from Customer Remarks, standardizing category naming conventions, normalizing text formatting, creating synthetic summaries and professional response templates, and removing records with extremely short or unusable remarks.

In [6]:
# Keep only relevant columns
complaints_df = complaints_df[[
    'Customer Remarks',
    'category',
    'Sub-category',
    'CSAT Score',
    'channel_name',
    'Product_category'
]]

# Remove rows missing complaint text
complaints_df = complaints_df.dropna(
    subset=['Customer Remarks']
)

# Remove very short remarks
complaints_df = complaints_df[
    complaints_df['Customer Remarks'].str.len() > 25
]

# Normalize text formatting
complaints_df['Customer Remarks'] = (
    complaints_df['Customer Remarks']
    .str.strip()
    .str.replace(r'\s+', ' ', regex=True)
)

complaints_df.head()

,Customer Remarks,category,Sub-category,CSAT Score,channel_name,Product_category
8,Shopzilla app and it's all coustomer care serv...,Returns,Missing,5,Inbound,NaN
25,Issue not resolved pathetic service,Order Related,Order status enquiry,1,Inbound,Mobile
30,Give proper guidance to the customer,Returns,Reverse Pickup Enquiry,5,Inbound,NaN
31,Please highlight if the products cannot be ret...,Order Related,Unable to track,1,Inbound,NaN
33,Please don't sale any destroy product or cheep...,Order Related,Installation/demo,5,Outcall,NaN


With those updates we should have a dataset that will be suitable for fine-tuning with improved consistency across training examples. We will now use this data to create a very small training dataset consisting of 25 records to reduce API training costs, speed up experimentation, validate workflow feasibility, and demonstrate the end-to-end tuning process. This approach is specific to demonstrating proof-of-concept, production deployment would preferably have an existing dataset with previously approved and consistent responses in alignment to organization requirements and previously received complaints. This will also mirror a pilot dataset that would be used prior to attempted scaling.

In [10]:
small_training_df = complaints_df.sample(
    n=25,
    random_state=12
)

small_training_df.head()

training_examples = []

for _, row in small_training_df.iterrows():

    example = {
        "messages": [
            {
                "role": "system",
                "content": (
                    "You are a financial institution complaint analyst specializing"
                    "in customer complaint categorization, summarization, and professional"
                    "response drafting."
                )
            },
            {
                "role": "user",
                "content": row['Customer Remarks']
            },
            {
                "role": "assistant",
                "content": json.dumps({
                    "category": row['category'],
                    "subcategory": row['Sub-category'],
                    "summary": (
                        f"Customer reported an issue "
                        f"regarding {row['Sub-category']}."
                    ),
                    "response": (
                        "Thank you for bringing this issue to our attention. "
                        "We understand your concern and are currently reviewing the matter. "
                        "Our support team will continue working toward a resolution as quickly as possible."
                    )
                })
            }
        ]
    }

    training_examples.append(example)

print(len(training_examples))

25


This step converts the data into supervised fine-tuning examples that are compatible with OpenAI's training format. Each example contains a system instruction, a user complaint, and a structured assistant response. We chose JSON-formatted outputs because one of the consistent issues that can be encountered during prompt experimentation is inconsistent formatiting. Choosing the JSON-formatting adn providing the training in structured outputs should improve the chance of response consistency, information inference, and allow for application integration.

A boiler plate response was used to create the file but this would not be accurate to each of the complaints and mirror the desired responses we would expect. After the file was created each individual example of the file was reviewed and updated with an appropriate response. This will mirror training data that would be present in an organization with previous complaints responses that would be retained and could be used for training.

In [11]:
with open("complaints_training.jsonl", "w") as f:

    for example in training_examples:
        f.write(json.dumps(example) + "\n")

OpenAI fine-tuning requires training data in JSONL format this will also allow us to return to the created training examples for repeated experimentation if needed. We can now upload the training dataset to OpenAI so it can be used during fine-tuning. We will receive the file ID, processing status, and metadata as identifiers.

In [7]:
upload_url = "https://api.openai.com/v1/files"

files = {
    "file": open("complaints_training.jsonl", "rb")
}

data = {
    "purpose": "fine-tune"
}

response = requests.post(
    upload_url,
    headers=headers,
    files=files,
    data=data
)

training_file = response.json()

print(training_file)

{'object': 'file', 'id': 'file-5mx7mCX4b42W3GyYvkyzq9', 'purpose': 'fine-tune', 'filename': 'complaints_training.jsonl', 'bytes': 17199, 'created_at': 1778167087, 'expires_at': None, 'status': 'processed', 'status_details': None}


We can now create the fine-tuning job. gpt-3.5-turbo was chosen to be more cost-efficient and allow training iterations to complete faster. The fine-tuning process will allow the model to learn the complaint categorization patterns, preferred response tone, and structured formatting behavior.

In [8]:
fine_tune_url = (
    "https://api.openai.com/v1/fine_tuning/jobs"
)

payload = {
    "training_file": training_file["id"],
    "model": "gpt-3.5-turbo"
}

response = requests.post(
    fine_tune_url,
    headers={
        **headers,
        "Content-Type": "application/json"
    },
    json=payload
)

fine_tune_job = response.json()

print(fine_tune_job)

{'object': 'fine_tuning.job', 'id': 'ftjob-E9sJykpDgVjfpjkbbeNflbXL', 'model': 'gpt-3.5-turbo-0125', 'created_at': 1778167098, 'finished_at': None, 'fine_tuned_model': None, 'organization_id': 'org-yQz0Gc4tnLoUjbOYucZMIYVL', 'result_files': [], 'status': 'validating_files', 'validation_file': None, 'training_file': 'file-5mx7mCX4b42W3GyYvkyzq9', 'hyperparameters': {'n_epochs': 'auto', 'batch_size': 'auto', 'learning_rate_multiplier': 'auto'}, 'trained_tokens': None, 'error': {}, 'user_provided_suffix': None, 'seed': 77625600, 'estimated_finish': None, 'integrations': [], 'metadata': None, 'usage_metrics': None, 'shared_with_openai': False, 'eval_id': None, 'internal_worker_backend': None, 'internal_peashooter_execution': None, 'train_experiment_id': None, 'eval_experiment_id': None, 'method': {'type': 'supervised', 'supervised': {'hyperparameters': {'batch_size': 'auto', 'learning_rate_multiplier': 'auto', 'n_epochs': 'auto'}}}}


We will monitor fine-tuning job progress via training status, progress information, fine-tuned model identifier, and validation information. This allows us to be aware of state and progress and ensure successful training is completed, and the time it takes can vary depending on queue load, dataset size, and model complexity.

In [9]:
job_id = fine_tune_job["id"]

status_url = (
    f"https://api.openai.com/v1/"
    f"fine_tuning/jobs/{job_id}"
)

job_status = requests.get(
    status_url,
    headers=headers
)

print(job_status.json())

{'object': 'fine_tuning.job', 'id': 'ftjob-E9sJykpDgVjfpjkbbeNflbXL', 'model': 'gpt-3.5-turbo-0125', 'created_at': 1778167098, 'finished_at': None, 'fine_tuned_model': None, 'organization_id': 'org-yQz0Gc4tnLoUjbOYucZMIYVL', 'result_files': [], 'status': 'validating_files', 'validation_file': None, 'training_file': 'file-5mx7mCX4b42W3GyYvkyzq9', 'hyperparameters': {'n_epochs': 'auto', 'batch_size': 'auto', 'learning_rate_multiplier': 'auto'}, 'trained_tokens': None, 'error': {}, 'user_provided_suffix': None, 'seed': 77625600, 'estimated_finish': None, 'integrations': [], 'metadata': None, 'usage_metrics': None, 'shared_with_openai': False, 'eval_id': None, 'internal_worker_backend': None, 'internal_peashooter_execution': None, 'train_experiment_id': None, 'eval_experiment_id': None, 'method': {'type': 'supervised', 'supervised': {'hyperparameters': {'n_epochs': 'auto', 'batch_size': 'auto', 'learning_rate_multiplier': 'auto'}}}}


In [10]:
status = "running"

while status not in ["succeeded", "failed"]:
    url = f"https://api.openai.com/v1/fine_tuning/jobs/{job_id}"
    response = requests.get(url, headers=headers)
    job_data = response.json()
    
    status = job_data["status"]
    print("Current Status:", status)
    
    time.sleep(10)

print("Final Status:", status)

Current Status: validating_files
Current Status: validating_files
Current Status: validating_files
Current Status: validating_files
Current Status: validating_files
Current Status: validating_files
Current Status: running
Current Status: running
Current Status: running
Current Status: running
Current Status: running
Current Status: running
Current Status: running
Current Status: running
Current Status: running
Current Status: running
Current Status: running
Current Status: running
Current Status: running
Current Status: running
Current Status: running
Current Status: running
Current Status: running
Current Status: running
Current Status: running
Current Status: running
Current Status: running
Current Status: running
Current Status: running
Current Status: running
Current Status: running
Current Status: running
Current Status: running
Current Status: running
Current Status: running
Current Status: running
Current Status: running
Current Status: running
Current Status: running
Current St

With fine-tuning training completed we can retrieve and evaluate the model results. In order to evaluate this we will submit a prompt to the base model and repeat that prompt with the fine-tuned model to evaluate response structure, tone, and information. The base model prompt will create a baseline for evaluating whether the fine-tuning actually improved performance. It is expected that the base model will perform well in understanding context and generating a professional response but there will likely be high variance in response structure.

In [11]:
completed_job = requests.get(
    status_url,
    headers=headers
).json()

fine_tuned_model = (
    completed_job['fine_tuned_model']
)

print(fine_tuned_model)

ft:gpt-3.5-turbo-0125:personal::DcutCBnR


In [12]:
base_model_url = (
    "https://api.openai.com/v1/chat/completions"
)

base_payload = {
    "model": "gpt-3.5-turbo",
    "messages": [
        {
            "role": "system",
            "content": (
                "You are a financial institution "
                "complaint analyst specializing "
                "in complaint categorization and "
                "professional response generation."
            )
        },
        {
            "role": "user",
            "content": (
                "The support team has not resolved "
                "my refund issue after multiple "
                "follow-ups."
            )
        }
    ]
}

base_response = requests.post(
    base_model_url,
    headers={
        **headers,
        "Content-Type": "application/json"
    },
    json=base_payload
)

base_result = base_response.json()

print(
    base_result['choices'][0]['message']['content']
)

I'm sorry to hear about the situation you're facing regarding your refund issue. As a financial institution, we understand the importance of timely and effective resolution for our customers. Rest assured, I will escalate your complaint to the relevant team immediately to ensure that your refund issue is resolved as soon as possible. Thank you for bringing this to our attention, and we apologize for any inconvenience this may have caused. If you have any additional information or details you would like to provide, please feel free to share them with us.


In [13]:
inference_url = (
    "https://api.openai.com/v1/chat/completions"
)

payload = {
    "model": fine_tuned_model,
    "messages": [
        {
            "role": "system",
            "content": (
                "You are a financial institution "
                "complaint analyst specializing "
                "in complaint categorization and "
                "professional response generation."
            )
        },
        {
            "role": "user",
            "content": (
                "The support team has not resolved "
                "my refund issue after multiple "
                "follow-ups."
            )
        }
    ]
}

response = requests.post(
    inference_url,
    headers={
        **headers,
        "Content-Type": "application/json"
    },
    json=payload
)

result = response.json()

print(
    result['choices'][0]['message']['content']
)

{"category": "Returns", "subcategory": "Delay related", "response": "Thank you for bringing this issue to our attention. We understand your concern and are currently reviewing the matter. Our support team will continue working toward a resolution as quickly as possible."}


As expected, the base model demonstrated strong general language understanding but did not have the required domain specific consistency we would require for production integration. We can see after fine-tuning is completed that there is a structured output that is now adopted compared to base response. The suggested response return is more concise as well compared to the based model which may be appropriate based on the amount of detail the organization wants to provide to a review currently in progress. Something that really is valuable here is the output predictability. The fine-tuned model consistency in both structure and tone is important in a regulated environment. There is still an opporunity for potential refinement in sub-category identification and more specific personalization in response structure.